# SALBA ML System - Notebook 1: Data Exploration

## Objective
Explore the training dataset to understand disaster patterns and prepare data for ML models.

## Steps
1. Load data from MongoDB/CSV
2. Exploratory Data Analysis (EDA)
3. Statistical analysis
4. Visualizations
5. Data quality assessment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded")

In [ ]:
# Load training data
df = pd.read_csv('data/training_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Dataset info
print("Dataset Info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nStatistical summary:")
df.describe()

In [ ]:
# Disaster type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
disaster_counts = df['disaster_type'].value_counts()
axes[0].bar(disaster_counts.index, disaster_counts.values, color='steelblue')
axes[0].set_title('Disaster Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(disaster_counts.values, labels=disaster_counts.index, autopct='%1.1f%%')
axes[1].set_title('Disaster Type Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nDisaster Type Breakdown:")
print(disaster_counts)

In [ ]:
# Severity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

severity_counts = df['severity'].value_counts()
severity_order = ['low', 'moderate', 'high', 'critical']
severity_counts = severity_counts.reindex([s for s in severity_order if s in severity_counts.index])

colors = ['green', 'yellow', 'orange', 'red']
axes[0].bar(severity_counts.index, severity_counts.values, color=colors[:len(severity_counts)])
axes[0].set_title('Severity Level Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# By disaster type
severity_by_type = pd.crosstab(df['disaster_type'], df['severity'])
severity_by_type.plot(kind='bar', stacked=False, ax=axes[1])
axes[1].set_title('Severity by Disaster Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].legend(title='Severity', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

print("\nSeverity Distribution:")
print(severity_counts)

In [ ]:
# Feature analysis
print("Feature Statistics:")
print(f"\nText Length:")
print(f"  Mean: {df['text_length'].mean():.2f}")
print(f"  Median: {df['text_length'].median():.2f}")
print(f"  Max: {df['text_length'].max():.0f}")

print(f"\nWord Count:")
print(f"  Mean: {df['word_count'].mean():.2f}")
print(f"  Median: {df['word_count'].median():.2f}")

print(f"\nUrgency Keywords:")
print(f"  With urgency: {df['has_urgency_keywords'].sum()} reports")
print(f"  Percentage: {(df['has_urgency_keywords'].sum() / len(df) * 100):.1f}%")

print(f"\nPrank Keywords:")
print(f"  With prank keywords: {df['has_prank_keywords'].sum()} reports")
print(f"  Percentage: {(df['has_prank_keywords'].sum() / len(df) * 100):.1f}%")

In [ ]:
# Geographic distribution
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['longitude'], df['latitude'], 
                       c=df['severity'].map({'low': 0, 'moderate': 1, 'high': 2, 'critical': 3}),
                       s=100, alpha=0.6, cmap='RdYlGn_r')
plt.colorbar(scatter, label='Severity Level')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Geographic Distribution of Disasters', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nGeographic Range:")
print(f"  Latitude: {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
print(f"  Longitude: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")

In [ ]:
# Time analysis
df['created_at'] = pd.to_datetime(df['created_at'])
df['hour'] = df['created_at'].dt.hour
df['month'] = df['created_at'].dt.month
df['day_of_week'] = df['created_at'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By hour
hourly = df['hour'].value_counts().sort_index()
axes[0].bar(hourly.index, hourly.values, color='skyblue')
axes[0].set_title('Reports by Hour of Day', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

# By month
monthly = df['month'].value_counts().sort_index()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar([months[m-1] for m in monthly.index], monthly.values, color='lightcoral')
axes[1].set_title('Reports by Month', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Class balance check
print("Class Balance Analysis:")
print("\nDisaster Type Balance:")
type_balance = df['disaster_type'].value_counts(normalize=True)
for disaster, pct in type_balance.items():
    print(f"  {disaster}: {pct*100:.1f}%")

print("\nSeverity Balance:")
sev_balance = df['severity'].value_counts(normalize=True)
for severity, pct in sev_balance.items():
    print(f"  {severity}: {pct*100:.1f}%")

# Imbalance warning
max_pct = type_balance.max() * 100
min_pct = type_balance.min() * 100
imbalance_ratio = max_pct / min_pct
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}x")
if imbalance_ratio > 3:
    print("⚠️  Data is imbalanced. Consider using class weighting in models.")

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("DATA EXPLORATION SUMMARY")
print("="*60)
print(f"Total samples: {len(df)}")
print(f"Features: {len(df.columns)}")
print(f"\nDisaster types: {df['disaster_type'].nunique()}")
print(f"Severity levels: {df['severity'].nunique()}")
print(f"\nDate range: {df['created_at'].min().date()} to {df['created_at'].max().date()}")
print(f"\n✅ Data exploration complete!")
print("\nNext steps:")
print("1. Run notebook 02 for feature engineering")
print("2. Then notebook 03 for model training")